# Chương 9 – Quản lý vòng đời model: Registry, Drift, Rollback, Safety (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phuongtnd1994/ptnd-ModelAdaptationBook/blob/main/code/chapter09/chapter09_colab.ipynb)

Stage 1, 2, 4 chạy trên CPU. Stage 3 (canary prompts) và 5 (safety monitor) cần GPU ~8–12 GB.
Bonus: dùng **Claude** đọc các report JSON và viết báo cáo vận hành.

## 0. Cài đặt môi trường

Chạy lần lượt các ô dưới đây **một lần** mỗi khi mở runtime mới. Ô đầu tiên clone repo, cài package của sách và tạo hàm `run()` để chạy lệnh và in log trực tiếp ra notebook.

In [ ]:
import os, sys, subprocess

# Repo chứa code của sách. Nếu fork của bạn để private, đổi sang repo gốc:
#   https://github.com/bahree/ModelAdaptationBook.git
REPO_URL = "https://github.com/phuongtnd1994/ptnd-ModelAdaptationBook.git"
REPO_DIR = "/content/ModelAdaptationBook"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ["PYTHONUNBUFFERED"] = "1"   # log của script con hiện ra ngay
os.environ["WANDB_DISABLED"] = "true"  # không dùng Weights & Biases trên Colab


def run(cmd: str) -> None:
    """Chạy một lệnh shell từ thư mục code/, in log trực tiếp, báo lỗi nếu thất bại."""
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {p.returncode}): {cmd}")


run('pip install -q -e ".[dev]"')
print("\nĐã cài xong. Thư mục làm việc:", os.getcwd())

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | {mem:.1f} GB | compute capability {cap[0]}.{cap[1]}")
    if cap < (8, 0):
        print("CẢNH BÁO: GPU này (ví dụ T4) không hỗ trợ bf16 gốc. Các bước TRAINING "
              "trong sách dùng bf16 -> hãy đổi sang L4 hoặc A100 "
              "(Runtime > Change runtime type).")
else:
    print("Không có GPU. Chỉ các bước chạy trên CPU / gọi API hoạt động được.")

### Khoá API (Colab Secrets)

Mở biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm:

- `ANTHROPIC_API_KEY` = API key Claude của bạn (**bắt buộc** cho notebook này)
- `HF_TOKEN` = token Hugging Face (không bắt buộc)

và bật *Notebook access*. Nếu chưa thêm, ô dưới sẽ hỏi bạn nhập tay.

In [ ]:
import os
from getpass import getpass


def get_secret(name, required=False):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    if required:
        return getpass(f"Nhập {name}: ")
    return None


os.environ["ANTHROPIC_API_KEY"] = get_secret("ANTHROPIC_API_KEY", required=True)
hf = get_secret("HF_TOKEN")  # không bắt buộc, giúp tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
print("Đã nạp khoá:", [k for k in ("ANTHROPIC_API_KEY", "HF_TOKEN") if os.environ.get(k)])

### Tải model đã train sẵn từ Hugging Face

Tác giả sách đã publish model của từng chương tại [`bahree/ModelAdaptationBook`](https://huggingface.co/bahree/ModelAdaptationBook). Ô dưới tải đúng các thư mục cần cho chương này về `/content/hub/` để các script dùng như đường dẫn local (không cần tự train các chương trước).

In [ ]:
from huggingface_hub import snapshot_download

HUB_DIR = "/content/hub"
SUBFOLDERS = ["ch6-sft", "ch8-dpo"]
snapshot_download("bahree/ModelAdaptationBook", local_dir=HUB_DIR,
                  allow_patterns=[f"{s}/*" for s in SUBFOLDERS])
for s in SUBFOLDERS:
    print(s, "->", f"{HUB_DIR}/{s}", sorted(os.listdir(f"{HUB_DIR}/{s}"))[:6])

In [ ]:
SFT_DIR = "/content/hub/ch6-sft"
DPO_DIR = "/content/hub/ch8-dpo"
REG = "chapter09/runs/colab_registry"   # registry riêng, không đụng file mẫu của sách

## 1. Unit test

In [ ]:
run("pytest chapter09/tests/ -q")

## 2. Stage 1 – Model registry (CPU)

Đăng ký 2 phiên bản, promote, rồi rollback.

In [ ]:
run(f"python -m chapter09.model_registry --registry_dir {REG} register --name it-support-v1 "
    f"--technique sft --base_model Qwen/Qwen3-4B-Instruct-2507 --data_hash abc123 "
    f"--checkpoint_path {SFT_DIR} --eval_metrics '{{\"overall_f1\": 0.168}}'")
run(f"python -m chapter09.model_registry --registry_dir {REG} promote --version_tag it-support-v1")
run(f"python -m chapter09.model_registry --registry_dir {REG} register --name it-support-v2 "
    f"--technique dpo --base_model Qwen/Qwen3-4B-Instruct-2507 --data_hash def456 "
    f"--checkpoint_path {DPO_DIR} --eval_metrics '{{\"overall_f1\": 0.170}}'")
run(f"python -m chapter09.model_registry --registry_dir {REG} promote --version_tag it-support-v2")
run(f"python -m chapter09.model_registry --registry_dir {REG} list")

In [ ]:
run(f"python -m chapter09.model_registry --registry_dir {REG} rollback")
run(f"python -m chapter09.model_registry --registry_dir {REG} list")

## 3. Stage 2 – Drift detection (CPU)

So dữ liệu train với (a) valid cùng phân phối và (b) dữ liệu production bị dịch chuyển.

In [ ]:
run("python -m chapter09.drift_detector --reference data/it_support/train.jsonl "
    "--production data/it_support/valid.jsonl --output chapter09/eval/drift_report.json")
run("python -m chapter09.scripts.build_drift_demo_sets")
run("python -m chapter09.drift_detector --reference data/it_support/train.jsonl "
    "--production chapter09/data/shifted_production.jsonl --output chapter09/eval/drift_report_shifted.json")

## 4. Stage 3 – Canary prompts (GPU)

Tạo baseline với SFT, rồi so model DPO với baseline.

In [ ]:
run(f"python -m chapter09.scripts.run_canary_prompts --model_dir {SFT_DIR} --output chapter09/eval/canary_baseline.jsonl")
run(f"python -m chapter09.scripts.run_canary_prompts --model_dir {DPO_DIR} "
    "--output chapter09/eval/canary_outputs.jsonl --baseline chapter09/eval/canary_baseline.jsonl")

## 5. Stage 4 – Rollback demo (CPU)

In [ ]:
run("python -m chapter09.rollback_demo --registry_dir chapter09/runs/colab_demo_registry --output chapter09/eval/rollback_report.json")

## 6. Stage 5 – Safety monitor (GPU)

In [ ]:
run(f"python -m chapter09.safety_monitor --model_dir {SFT_DIR} --output chapter09/eval/safety_report.json")
run(f"python -m chapter09.safety_monitor --model_dir {DPO_DIR} "
    "--output chapter09/eval/safety_report_dpo.json --baseline chapter09/eval/safety_report.json")

## 7. (Bonus) Claude viết báo cáo vận hành

Gom các report JSON và nhờ Claude tóm tắt: có drift không, có nên promote DPO, rủi ro an toàn.

In [ ]:
import anthropic, json, pathlib

reports = {}
for name in ["drift_report.json", "drift_report_shifted.json", "rollback_report.json",
             "safety_report.json", "safety_report_dpo.json"]:
    p = pathlib.Path("chapter09/eval") / name
    if p.exists():
        reports[name] = json.loads(p.read_text())

client = anthropic.Anthropic()
resp = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=8000,
    messages=[{"role": "user", "content":
        "Bạn là MLOps engineer. Dưới đây là các report giám sát của một model IT-support "
        "(SFT là bản đang chạy, DPO là ứng viên). Viết báo cáo ngắn bằng tiếng Việt: "
        "1) có drift dữ liệu không, 2) DPO có an toàn để promote không, 3) hành động đề xuất.\n\n"
        + json.dumps(reports, ensure_ascii=False)[:150000]}],
)
print("".join(b.text for b in resp.content if b.type == "text"))

## Lưu kết quả về Google Drive (tuỳ chọn)

Runtime Colab sẽ bị xoá khi ngắt kết nối. Chạy ô dưới để copy thư mục kết quả sang Drive.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/ModelAdaptationBook"
for p in ["chapter09/eval", "chapter09/runs"]:
    src = os.path.join(CODE_DIR, p)
    dst = os.path.join(DEST, p)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Đã lưu", p, "->", dst)
    elif os.path.isfile(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Đã lưu", p, "->", dst)